<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 2.4: 时序逻辑
**上一节: [控制流](2.3_control_flow.ipynb)**<br>
**下一节: [FIR滤波器](2.5_exercise.ipynb)**

## 动机
没有状态，你无法编写任何有意义的数字逻辑。没有状态，你无法编写任何有意义的数字逻辑。没有状态，你无法编写任何有意义的数字逻辑....

明白了吗？因为如果不存储中间结果，你将无法取得任何进展。

好吧，抛开这个糟糕的笑话，本模块将描述如何在 Chisel 中表达常见的时序模式。到本模块结束时，你应该能够在 Chisel 中实现和测试移位寄存器。

需要强调的是，本节可能不会给你留下深刻印象。Chisel 的强大之处不在于新的时序逻辑模式，而在于设计的参数化。在我们展示这种能力之前，我们必须了解这些时序模式是什么。因此，本节将向你展示 Chisel 可以完成 Verilog 可以完成的大部分工作 - 你只需要学习 Chisel 语法。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 寄存器
Chisel 中基本的有状态元素是寄存器，用 `Reg` 表示。
`Reg` 保持其输出值直到时钟的上升沿，此时它采用其输入的值。
默认情况下，每个 Chisel `Module` 都有一个隐式时钟，设计中的每个寄存器都使用该时钟。
这使您不必总是在代码中指定相同的时钟。

<span style="color:blue">**示例：使用寄存器**</span><br>
以下代码块实现了一个模块，该模块获取输入，加 1，然后将其连接为寄存器的输入。
*注意：对于多时钟设计，可以覆盖隐式时钟。有关示例，请参见附录。*

In [ ]:
class RegisterModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(12.W))
    val out = Output(UInt(12.W))
  })
  
  val register = Reg(UInt(12.W))
  register := io.in + 1.U
  io.out := register
}

test(new RegisterModule) { c =>
  for (i <- 0 until 100) {
    c.io.in.poke(i.U)
    c.clock.step(1)
    c.io.out.expect((i + 1).U)
  }
}
println("SUCCESS!!")

寄存器通过调用 `Reg(tpe)` 创建，其中 `tpe` 是一个编码我们所需寄存器类型的变量。
在这个例子中，`tpe` 是一个 12 位的 `UInt`。

看看上面的测试器在做什么。
在调用 `poke()` 和 `expect` 之间，有一个对 `step(1)` 的调用。
这告诉测试框架使时钟跳动一次，这将导致寄存器将其输入传递到输出。

调用 `step(n)` 将使时钟跳动 `n` 次。

敏锐的观察者会注意到，之前测试组合逻辑的测试器没有调用 `step()`。这是因为在输入上调用 `poke()` 会立即通过组合逻辑传播更新后的值。调用 `step()` 仅需要更新时序逻辑中的状态元素。

下面的代码块将显示 `RegisterModule` 生成的 Verilog。

注意：
* 模块有一个时钟（和复位）输入，你没有添加 - 这是隐式时钟
* 变量 `register` 显示为 `reg [11:0]`，如预期所示
* 有一个由 `ifdef Randomize` 分隔的块，在仿真开始前将寄存器初始化为某个随机变量
* `register` 在 `posedge clock` 时更新

In [ ]:
println(getVerilog(new RegisterModule))

一个重要注意事项是，Chisel 区分类型（如 `UInt`）和硬件节点（如字面量 `2.U`，或 `myReg` 的输出）。虽然
```scala
val myReg = Reg(UInt(2.W))
```
是合法的，因为 Reg 需要一个数据类型作为模型，
```scala
val myReg = Reg(2.U)
```
是错误的，因为 `2.U` 已经是一个硬件节点，不能用作模型。

<span style="color:blue">**示例：RegNext**</span><br>
Chisel 有一个方便的寄存器对象，用于具有简单输入连接的寄存器。之前的 `Module` 可以缩短为以下 `Module`。注意这次我们不需要指定寄存器位宽。它从寄存器的输出连接推断出来，在这种情况下是 `io.out`。

In [ ]:
class RegNextModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(12.W))
    val out = Output(UInt(12.W))
  })
  
  // register bitwidth is inferred from io.out
  io.out := RegNext(io.in + 1.U)
}

test(new RegNextModule) { c =>
  for (i <- 0 until 100) {
    c.io.in.poke(i.U)
    c.clock.step(1)
    c.io.out.expect((i + 1).U)
  }
}
println("SUCCESS!!")

Verilog 看起来几乎和以前一样，尽管寄存器名称是生成的而不是明确定义的。

In [ ]:
println(getVerilog(new RegNextModule))

---
# `RegInit`

`RegisterModule` 中的寄存器被初始化为随机数据以进行仿真。
除非另有指定，寄存器没有复位值（或复位）。
创建复位到给定值的寄存器的方法是使用 `RegInit`。

例如，一个初始化为零的 12 位寄存器可以用以下方式创建。
以下两个版本都是有效的，并且做同样的事情：
```scala
val myReg = RegInit(UInt(12.W), 0.U)
val myReg = RegInit(0.U(12.W))
```

第一个版本有两个参数。
第一个参数是一个类型节点，指定数据类型及其宽度。
第二个参数是一个硬件节点，指定复位值，在这种情况下为 0。

第二个版本有一个参数。
它是一个指定复位值的硬件节点，但通常是 `0.U`。

<span style="color:blue">**示例：初始化的寄存器**</span><br>
以下演示了使用 `RegInit()`，初始化为零。

In [ ]:
class RegInitModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(12.W))
    val out = Output(UInt(12.W))
  })
  
  val register = RegInit(0.U(12.W))
  register := io.in + 1.U
  io.out := register
}

println(getVerilog(new RegInitModule))

注意，生成的 Verilog 现在有一个检查 `if (reset)` 的块，用于将寄存器复位到 0。
还要注意，这是在 `always @(posedge clock)` 块内部。
Chisel 的隐式复位是高电平有效且同步的。
在调用复位之前，寄存器仍然被初始化为随机垃圾值。
`PeekPokeTesters` 在运行测试之前总是调用复位，但你也可以使用 `reset(n)` 函数手动调用复位，其中复位高电平持续 `n` 个周期。

时钟在 Chisel 中有自己的类型（`Clock`），应该这样声明。
*`Bool` 可以通过在其上调用 `asClock()` 转换为 `Clock`，但你应该小心不要做愚蠢的事情。*

还要注意，`chisel-testers` 目前不完全支持多时钟设计。

<span style="color:blue">**示例：多时钟模块**</span><br>
一个具有多个时钟和复位信号的模块。

In [ ]:
class FindMax extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(10.W))
    val max = Output(UInt(10.W))
  })

  val max = RegInit(0.U(10.W))
  when (io.in > max) {
    max := io.in
  }
  io.max := max
}

test(new FindMax) { c =>
    c.io.max.expect(0.U)
    c.io.in.poke(1.U)
    c.clock.step(1)
    c.io.max.expect(1.U)
    c.io.in.poke(3.U)
    c.clock.step(1)
    c.io.max.expect(3.U)
    c.io.in.poke(2.U)
    c.clock.step(1)
    c.io.max.expect(3.U)
    c.io.in.poke(24.U)
    c.clock.step(1)
    c.io.max.expect(24.U)
}
println("SUCCESS!!")

---
# 其他寄存器示例

对寄存器调用的操作是在寄存器的**输出**上执行的，操作的类型取决于寄存器的类型。
这意味着你可以写
```scala
val reg: UInt = Reg(UInt(4.W))
```
这意味着值 `reg` 的类型是 `UInt`，你可以做通常对 `UInt` 可以做的事情，比如 `+`、`-` 等。


你不限于将 `UInt` 与寄存器一起使用，你可以使用基类型 `chisel3.Data` 的任何子类。这包括用于有符号整数的 `SInt` 和许多其他类型。

<span style="color:blue">**示例：梳状滤波器**</span><br>
以下示例展示了一个梳状滤波器。

In [ ]:
class Comb extends Module {
  val io = IO(new Bundle {
    val in  = Input(SInt(12.W))
    val out = Output(SInt(12.W))
  })

  val delay: SInt = Reg(SInt(12.W))
  delay := io.in
  io.out := io.in - delay
}
println(getVerilog(new Comb))

---
# 练习
<span style="color:red">**练习：移位寄存器**</span><br>
根据你新学到的寄存器知识，构建一个实现 LFSR 移位寄存器的模块。具体来说：

In [ ]:
class MyShiftRegister(val init: Int = 1) extends Module {
  val io = IO(new Bundle {
    val in  = Input(Bool())
    val out = Output(UInt(4.W))
  })

  val state = RegInit(UInt(4.W), init.U)

  ???
}

test(new MyShiftRegister()) { c =>
  var state = c.init
  for (i <- 0 until 10) {
    // poke in LSB of i (i % 2)
    c.io.in.poke(((i % 2) != 0).B)
    // update expected state
    state = ((state * 2) + (i % 2)) & 0xf
    c.clock.step(1)
    c.io.out.expect(state.U)
  }
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val nextState = (state << 1) | io.in
  state := nextState
  io.out := state
</pre></article></div></section></div>

<span style="color:red">**练习：参数化移位寄存器（可选）**</span><br>
编写一个移位寄存器，其延迟（`n`）、初始值（`init`）可参数化，并且还有一个使能输入信号（`en`）。

In [ ]:
// n is the output width (number of delays - 1)
// init state to init
class MyOptionalShiftRegister(val n: Int, val init: BigInt = 1) extends Module {
  val io = IO(new Bundle {
    val en  = Input(Bool())
    val in  = Input(Bool())
    val out = Output(UInt(n.W))
  })

  val state = RegInit(init.U(n.W))

  ???
}

// test different depths
for (i <- Seq(3, 4, 8, 24, 65)) {
  println(s"Testing n=$i")
  test(new MyOptionalShiftRegister(n = i)) { c =>
    val inSeq = Seq(0, 1, 1, 1, 0, 1, 1, 0, 0, 1)
    var state = c.init
    var i = 0
    c.io.en.poke(true.B)
    while (i < 10 * c.n) {
      // poke in repeated inSeq
      val toPoke = inSeq(i % inSeq.length)
      c.io.in.poke((toPoke != 0).B)
      // update expected state
      state = ((state * 2) + toPoke) & BigInt("1"*c.n, 2)
      c.clock.step(1)
      c.io.out.expect(state.U)

      i += 1
    }
  }
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-2" />
<label for="check-2"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val nextState = (state << 1) | io.in
  when (io.en) {
    state  := nextState
  }
  io.out := state
</pre></article></div></section></div>

---
# Appendix: Explicit clock and reset
Chisel modules have a default clock and reset that are implicitly used by every register created inside them.
There are times where you want to be able to override this default behavior; perhaps you have a black box that generates a clock or reset signal, or you have a multi-clock design.

Chisel 提供了处理这些情况的构造。
时钟和复位可以分别或一起使用 `withClock() {}`、`withReset() {}` 和 `withClockAndReset() {}` 进行覆盖。
以下代码块将给出使用这些函数的示例。

需要注意的一点是，`reset`（截至本教程编写时）始终是同步的且类型为 `Bool`。
时钟在 Chisel 中有自己的类型（`Clock`），应该这样声明。
*`Bool` 可以通过调用 `asClock()` 转换为 `Clock`，但你应该小心不要做愚蠢的事情。*

还要注意 `chisel-testers` 目前对多时钟设计的支持不完整。

<span style="color:blue">**Example: Multi-Clock Module**</span><br>
A module with multiple clocks and reset signals.

In [ ]:
// we need to import multi-clock features
import chisel3.experimental.{withClock, withReset, withClockAndReset}

class ClockExamples extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(10.W))
    val alternateReset    = Input(Bool())
    val alternateClock    = Input(Clock())
    val outImplicit       = Output(UInt())
    val outAlternateReset = Output(UInt())
    val outAlternateClock = Output(UInt())
    val outAlternateBoth  = Output(UInt())
  })

  val imp = RegInit(0.U(10.W))
  imp := io.in
  io.outImplicit := imp

  withReset(io.alternateReset) {
    // everything in this scope with have alternateReset as the reset
    val altRst = RegInit(0.U(10.W))
    altRst := io.in
    io.outAlternateReset := altRst
  }

  withClock(io.alternateClock) {
    val altClk = RegInit(0.U(10.W))
    altClk := io.in
    io.outAlternateClock := altClk
  }

  withClockAndReset(io.alternateClock, io.alternateReset) {
    val alt = RegInit(0.U(10.W))
    alt := io.in
    io.outAlternateBoth := alt
  }
}

println(getVerilog(new ClockExamples))

---
# 总结
完成本节很棒！！你现在已经学会了如何在 Chisel 中创建寄存器和编写时序逻辑，这意味着你有足够的基本构建块来编写真实的电路。

下一节将把我们学到的所有内容结合到一个例子中！如果你需要一点鼓励，只需记住这位 Chisel 专家用户的话：

![BobRoss](http://i.qkme.me/3qbd5u.jpg)

---
# 你完成了！

[返回顶部。](#top)